# Day 15 · 跑通第一次 LoRA SFT

**配套讲义**: [`days/day-15.md`](../days/day-15.md) ｜ **需要 GPU（云机器）**

把训练真正跑起来 —— 先用 `--dry-run` 验数据与显存，再正式开训，亲眼看到 loss 在 50 步内开始下降，checkpoint 落盘。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w3.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys, torch
print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cuda   :", torch.version.cuda, "| available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"gpu    : {p.name}  {p.total_memory / 1024**3:.0f} GB")
    print("bf16   :", torch.cuda.is_bf16_supported())
else:
    print("⚠️  没有 GPU —— 这一天的训练/推理跑不了。先看 docs/13-hardware-and-cost.md 租机器")

## 1. 先看配置，再看数据

In [ ]:
import yaml, json, itertools
from pathlib import Path

cfg = yaml.safe_load(Path("../configs/sft_lora_3b.yaml").read_text())
for k, v in cfg.items():
    print(f"{k:22s} {v}")

## 2. 抽一个 batch 肉眼检查 label mask

**这是今天最值钱的一步**：亲眼看 `-100` 落在了哪里。

In [ ]:
import json
from pathlib import Path

p = Path("../data/processed/sft_train.jsonl")
rows = [json.loads(l) for l in p.read_text().splitlines() if l.strip()][:2]

for r in rows:
    print("=" * 70)
    print("图片:", r.get("images"))
    convo = r.get("conversations") or r.get("messages") or []
    for turn in convo:
        print(f"[{turn.get('from', turn.get('role'))}] {str(turn.get('value', turn.get('content')))[:120]}")

## 3. dry-run（先别烧 GPU）

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "src.train.sft_peft",
                    "--config", "configs/sft_lora_3b.yaml", "--dry-run"],
                   capture_output=True, text=True, cwd="..")
print(r.stdout[-3000:] or r.stderr[-3000:])

## 4. 正式开训

**在终端里跑，不要在 notebook 里跑** —— 训练要几个小时，notebook 断了就白跑。
```bash
cd /root/autodl-tmp/multimodal-lab
nohup python -m src.train.sft_peft --config configs/sft_lora_3b.yaml \
      > outputs/train_w3.log 2>&1 &
tail -f outputs/train_w3.log
```
用 `nohup` + `tail -f`，这样 SSH 断了训练还在。

## 验收清单

- [ ] `--dry-run` 通过，且打印的 visual token 数与 Day 4 算的一致
- [ ] 正式训练启动，`loss` 在前 50 步内开始下降（不必降到很低，只要趋势向下）
- [ ] `outputs/qwen25vl3b-cx-lora-v0/` 下出现 `adapter_model.safetensors`
- [ ] 能解释日志三条曲线的**正常形态**：loss 缓降、lr 先升后降（warmup+cosine）、grad_norm 平稳

**卡住了？** 回看 [`days/day-15.md`](../days/day-15.md) 第五节「容易踩的坑」。

> **明天**：`days/day-16.md` —— 读日志、制造 bug、画三联图